# 🧠 CerebroSpheroid V5: High-Performance Colab Training & Cloud Sync
### Complete Graded Developmental Curriculum & Real-Time Bioflo.in Sync Hub

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Colab_Training.ipynb)

**Key Highlights:**
- **Fast GPU/TPU Training:** Runs on Google Colab Free T4, L4, or A100 GPUs.
- **Two-Way Cloud Sync:** Continuous live checkpoint synchronization with `https://bioflo.in/api/cerebro/sync.php` via chunked HTTP range streaming (bypasses Cloudflare limits).
- **Zero Data Loss:** Automatically pulls the latest checkpoint from the cloud before training begins and uploads checkpoints asynchronously in the background.
- **Standardized Cerebro-1000 Benchmark:** Evaluates language, knowledge, math, code, and agentic tool dispatch.

## 1. Verify GPU Hardware Acceleration & System Environment

In [ ]:
import torch, os, sys
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Running on CPU! Go to Runtime -> Change runtime type -> Select T4 GPU or A100 for fast training.")

!nvidia-smi

## 2. Clone CerebroSpheroid V5 Repository & Install Dependencies

In [ ]:
%cd /content
if not os.path.exists("/content/cerebro_spheroid_v5"):
    !git clone https://github.com/AjitKumar-git/cerebro_spheroid_v5.git /content/cerebro_spheroid_v5
else:
    %cd /content/cerebro_spheroid_v5
    !git pull origin main

%cd /content/cerebro_spheroid_v5

# Install minimal fast training prerequisites
!pip install -q transformers datasets accelerate sentencepiece pyarrow

## 3. Verify Cloud Hub Connection & Pull Latest Checkpoint
Checks the status on `https://bioflo.in/api/cerebro/sync.php` and restores the most recent checkpoint.

In [ ]:
from src.cloud_sync import check_remote_manifest, pull_checkpoint

manifest = check_remote_manifest()
print(f"Sync Hub Status: {manifest.get('status')}")
print(f"Has Latest Checkpoint: {manifest.get('has_latest')}")
if manifest.get('has_latest'):
    latest_info = manifest.get('manifest', {}).get('latest', {})
    print(f"  • Step: {latest_info.get('step')}")
    print(f"  • Loss: {latest_info.get('loss')}")
    print(f"  • Updated: {latest_info.get('updated_at')}")

os.makedirs("/content/cerebro_spheroid_v5/checkpoints/graded_curriculum", exist_ok=True)
dest_path = "/content/cerebro_spheroid_v5/checkpoints/graded_curriculum/latest.pt"
if manifest.get('has_latest'):
    pull_checkpoint(dest_path, target_type="latest")

## 4. Launch Graded Curriculum Pre-Training with Live Cloud Sync
Trains CerebroSpheroid V5 across the developmental curriculum.
- Asynchronously uploads `latest.pt` and `best.pt` to `bioflo.in` in the background.
- Uses Adafactor optimizer for a 15 MB optimizer footprint.

In [ ]:
# Run training with live cloud synchronization
# Adjust --total-steps or --batch-size as needed for your Colab session
!python3 scripts/train_v5_graded_curriculum.py \
    --total-steps 1000000 \
    --batch-size 4 \
    --grad-accum 8 \
    --seq-len 64 \
    --max-lr 3e-4 \
    --save-interval 250 \
    --cloud-sync \
    --cloud-sync-interval 100

## 5. Evaluate on Cerebro-1000 Benchmark
Runs objective automated evaluation against the 1,000-prompt benchmark across all 5 cognitive pillars.

In [ ]:
# Run a 100-sample benchmark pulse check
!python3 eval/run_benchmark.py \
    --checkpoint checkpoints/graded_curriculum/latest.pt \
    --samples 100

# Display Markdown Benchmark Report
from IPython.display import display, Markdown
if os.path.exists("eval/results/BENCHMARK_REPORT.md"):
    with open("eval/results/BENCHMARK_REPORT.md") as f:
        display(Markdown(f.read()))